In [ ]:
import os
import openai
import chromadb
from sentence_transformers import SentenceTransformer
from retrieve_rerank import retrieve_and_rerank, format_context

# Support agent to answer user queries
class SupportAgent:
    def __init__(self, collection_name: str="vela_support"):
        self.client = openai.OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key= os.getenv("OPENROUTER_API_KEY")
        )

        # Initialize the embedding model and the ChromaDB collection which will be used for retrieval and rerank
        self.embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
        self.collection = chromadb.PersistentClient(path="./chroma_db").get_collection(collection_name)

        # Initialize the history of the current conversation since the model's memory
        # will refresh every new conversation
        self.history = []

        # Defining the tools that the agent can use to answer user queries
        self.toools = [
            {
                "name" : "search_knowledge_base", # Tool to search Vela Support knowledge base
                "description" : (
                    "Search the Vela support database for relevant information about products, pricing, "
                    "and troubleshooting. Use this whenever a customer asks a question that is specific to Vela's"
                    "features, plans, setup or commodities. Aim to provide the most relevant information from"
                    "the knowledge base to the customer."
                ),
                "input_schema" : {
                    "type" : "object",
                    "properties" : {
                        "query" : {
                            "type" : "string",
                            "description" : (
                                "The search query. Phrase this as a natural question on a topic using"
                                "single, plain and easily-understandable english."
                            )
                        }
                    },
                    "required" : ["query"]
                }
            },
            {
                "name" : "web_search",
                "description" : (
                    "Search the web for current information about Vela's products, pricing, and troubleshooting"
                    "outside Vela's knowledge base. Use this for general questions irrelevant to Vela's services"
                    "or when needing real-time information outside Vela's services."
                ),

                "input_schema" : {
                    "type" : "object",
                    "properties" : {
                        "query" : {
                            "type" : "string",
                            "description" : (
                                "The web search query . Phrase this as a natural question on a topic using"
                                "single, plain and easily-understandable english to get the clearest and most"
                                "direct results from the World Wide Web."
                            )
                        }
                    },
                    "required" : ["query"]
                }
            }
        ]